# 7. Phân tích thống kê cuối cùng

So sánh `XGBoost` (tham chiếu) với `Random Forest`, `Logistic Regression`, `Decision Tree`, `KNN`,
`Naive Bayes` trên `results/scores_all.csv` - 6 mô hình × 5 fold **dùng chung**. Chênh lệch ghép cặp =
`log loss XGBoost - log loss mô hình so sánh` (âm ⇒ XGBoost tốt hơn); Bonferroni cho họ 5 phép so sánh
(alpha 0.05 → 0.01).

## 7.1. Thiết lập môi trường

Chỉ đọc kết quả đã freeze, không train lại. Mục 7.6 - 7.7 dùng artifact sinh trong env authoritative
`dynamic` (Python 3.9.18, xgboost 2.1.4, pin ở `requirements.lock.txt`).

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  # notebooks/ -> gốc dự án
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import json

import pandas as pd
from IPython.display import Image

from src import final_analysis

## 7.2. Nạp và kiểm tra bảng điểm cuối

`validate_scores()` chặn trước mọi phân tích: đúng schema, 30 dòng (6 mô hình × 5 fold), không trùng
`model × fold`, không NaN/Inf, `log_loss >= 0`, `accuracy`/`macro_f1` ∈ `[0, 1]`. Sai là `raise`.

In [ ]:
scores = pd.read_csv(project_root / "results" / "scores_all.csv")
score_validation = final_analysis.validate_scores(scores)
score_validation

## 7.3. Bảng tổng hợp hiệu năng (mean ± std)

Trung bình ± độ lệch chuẩn qua 5 fold, sắp theo log loss - **chỉ số chính**.

In [ ]:
performance_summary = scores.groupby("model")[["log_loss", "accuracy", "macro_f1"]].agg(["mean", "std"])
performance_summary.sort_values(("log_loss", "mean"))

## 7.4. Kiểm định ghép cặp, khoảng tin cậy và hiệu chỉnh Bonferroni

Fold dùng chung nên chênh lệch ghép được theo từng fold: `paired t-test` + `CI 95%` (phân phối t), rồi
Bonferroni cho 5 phép so sánh. CI **hoàn toàn dưới 0** ⇒ ủng hộ XGBoost log loss thấp hơn; CI **chứa 0**
⇒ chưa đủ bằng chứng, **không** phải tương đương.

In [ ]:
pairwise = final_analysis.run_pairwise_comparisons(scores)
pairwise

## 7.5. Kiểm tra giả định và phân tích độ nhạy

`Shapiro-Wilk` chọn kiểm định chính: `p >= 0.05` giữ `paired t-test`, ngược lại chuyển `Wilcoxon
signed-rank` hai phía (exact). Bảng báo cáo cả hai p-value để thấy độ nhạy của kết luận; Bonferroni áp
lên kiểm định chính.

In [ ]:
assumption_checks = final_analysis.run_assumption_checks(scores)
assumption_checks

## 7.6. OOF authoritative của XGBoost và cổng tái tạo

Xác suất OOF lấy từ `results/oof/oof_xgboost_track_c.csv` (sinh trong env `dynamic`). Bản
`results/diagnostics/oof_xgboost_reproduction_attempt.csv` đã **SUPERSEDED** - sinh từ env khác, trượt
cổng 5/5 fold, chỉ giữ làm bằng chứng và **không** được dùng cho số báo cáo (xem
`report/track_c_provenance_audit.md`).

`validate_oof_xgboost()`: 9.600 dòng, `id` duy nhất, fold khớp `fold_id.csv`, xác suất ∈ `[0, 1]`, tổng
hàng ≈ 1. Cổng tái tạo so metric tính lại từ OOF với `results/scores_track_c.csv` theo từng fold.

In [ ]:
oof = pd.read_csv(project_root / final_analysis.AUTHORITATIVE_OOF)
official_folds = pd.read_csv(project_root / "data" / "interim" / "fold_id.csv")
oof_validation = final_analysis.validate_oof_xgboost(oof, official_folds)

gate = json.loads((project_root / final_analysis.AUTHORITATIVE_GATE).read_text())
reproduction = pd.read_csv(project_root / final_analysis.AUTHORITATIVE_PER_FOLD)
oof_validation, gate, reproduction

## 7.7. Phân tích calibration của mô hình tốt nhất

Reliability diagram + ECE cho mô hình có log loss thấp nhất (XGBoost) - log loss thưởng cho xác suất
được hiệu chỉnh tốt nên phải kiểm tra trực tiếp.

`run_calibration_analysis()` **raise `RuntimeError`** nếu cổng 7.6 chưa `PASS`, và là **nguồn duy nhất**
của các số calibration (notebook không tính lại inline ⇒ không có đường code thứ hai để lệch). ECE phụ
thuộc cách chia bin nên luôn kèm bảng độ nhạy `(5, 10, 15, 20)`.

In [ ]:
calibration_metrics = final_analysis.run_calibration_analysis(project_root, n_bins=10)

bin_sensitivity = pd.DataFrame(calibration_metrics.pop("bin_sensitivity"))
print({key: value for key, value in calibration_metrics.items()})
print("\nECE theo số bin:")
print(bin_sensitivity.to_string(index=False))

top_bins = pd.read_csv(project_root / final_analysis.CALIBRATION_TOP_LABEL_BINS_TABLE)
top_bins

In [ ]:
Image(filename=str(project_root / final_analysis.CALIBRATION_FIGURE))

## 7.8. Nhận xét

- **Xếp hạng log loss:** XGBoost **0.382** < RF 0.406 < LR 0.441 < DT 0.474 < KNN 0.660 < NB 2.211.
- **Ghép cặp (7.4):** cả 5 CI 95% nằm hoàn toàn dưới 0, cả 5 p-value Bonferroni `< 0.01` ⇒ ủng hộ XGBoost
  có log loss thấp hơn từng mô hình so sánh.
- **Kiểm định chính (7.5):** RF/DT/KNN/NB giữ paired t-test, đều có ý nghĩa sau Bonferroni. Riêng
  **Logistic Regression** có Shapiro `p ≈ 0.01423` → Wilcoxon: `p thô = 0.0625`, `p hiệu chỉnh = 0.3125`
  ⇒ **chưa đủ bằng chứng**, cũng **không** kết luận tương đương.
- **Vì sao t-test và Wilcoxon lệch nhau:** `n = 5` nên Shapiro power rất thấp, còn Wilcoxon hai phía exact
  có p nhỏ nhất là `0.0625` - không thể chạm alpha hiệu chỉnh 0.01. Tập train các fold chồng lấn ⇒ đọc
  p-value mức fold thận trọng.
- **Tái tạo (7.6):** gate `PASS` 5/5 fold (sai khác metric tối đa ~5.6e-17) ⇒ được phép chạy calibration.
- **Calibration (7.7):** top-label ECE ≈ **0.0117**, macro classwise ECE ≈ **0.0085**, Brier ≈ **0.2162**
  (10 bin); ECE chỉ tăng tới 0.0144 ở 20 bin - vẫn nhỏ.

## 7.9. Hạn chế

**So sánh thống kê:**
- Protocol chốt **sau khi đã có kết quả**, không preregistered.
- Chọn hyperparameter và đánh giá dùng lại **cùng bộ fold**, không nested CV.
- 5 fold ⇒ power rất thấp cho kiểm tra phân phối; tập train các fold chồng lấn ⇒ yếu giả định độc lập.

**Calibration** (đầy đủ ở `report/calibration_analysis.md`):
- Encoder/scaler fit **một lần trước** CV ⇒ tiền xử lý rò rỉ qua fold, calibration có thể lạc quan.
- Lớp `CL` chỉ 254/9.600 dòng (2,65%): classwise ECE đẹp nhưng bị chi phối bởi bin gần 0 ⇒ **không** phải
  bằng chứng lớp hiếm được hiệu chỉnh tốt.
- Bin top-label dưới 0.3 rỗng về cấu trúc (3 lớp ⇒ xác suất lớn nhất luôn `>= 1/3`).
- ECE phụ thuộc cách chia bin (xem 7.7); và đây là OOF của tập train, **không** phải test set cuộc thi.
- Không có calibrator nào được fit rồi đánh giá trên cùng bộ OOF.